# A Surface Mass Balance (SMB) emulator over Antarctica


This notebook is a demonstrator from the **Digital Twin Component for Ice Sheets (DTC-IS)**, an ESA-funded, EO-driven digital twin project.

DTC-IS integrates Earth Observation, in-situ data, reanalysis products, and advanced data-driven models to provide a dynamic and science-based representation of the Greenland and Antarctic ice sheets, including their ice shelves. It supports the monitoring, understanding, and prediction of ice-sheet change, delivering actionable information for research, policy, and scenario-based exploration.

[Project Website](https://dtc-ice-sheets.org/) · [Interactive Dashboard](https://dashboards.dtc-ice-sheets.org/) · [Query API](https://query.dtc-ice-sheets.org)


## Climate Super Resolution

Global Climate Models (GCM) cover the whole planet but do not contain fine spatial details, 
whereas Regional Climate Models (RCMs) provide highly detailed information but cover small areas and come at 
a high additional computational cost. This module shows how machine learning can be used to emulate high resolution 
regional climate models having as input coarse global climate models, offering a cost efficient alternative to dynamical
downscaling.

Based on the work of Van der Meer et al. ([link to the paper](https://doi.org/10.1029/2022MS003593)), the model provided
here can predict the Surface Mass Balance (SMB) of Antarctica at high resolution, starting from eight selected variables of ERA5 monthly means.
The pretrained model is a UNet, that can be selected in its basic configuration, or with attention mechanism.

---

**Key features:**
- **Location:** Choose a point on the map, which will be the center of the analysed tile.
- **Analysis period:** Specify the start and end date within the ERA5 availability window (typically 1940–present).
- **Animation:** Visualise the predicted Surface Mass Balance (SMB) evolving over time.
- **Evaluate:** When ground truth data is present, check the model performance for a given prediction.

**How to use this notebook:**
1. Run the setup cells below.
2. **Select a point** on the map, around which a tile will be created.
3. Select a **time window**, for which to predict the SMB.
4. Explore the predictions over time, and their relation with the inputs and with the ground truth

**Requirements:**
- Jupyter, VS Code with the Jupyter extension, or Google Colab.
- Internet access to the DTC-IS API.
- A DTC-IS API token (you will be asked for it below).

---

## Notebook preparations

In [ ]:
import sys

if "google.colab" in sys.modules:
    print("Installing dependencies...")
    %pip install -q xarray zarr netcdf4 cartopy fsspec aiohttp plotly seaborn ipympl ipywidgets==7.7.1 \
        dtc-query-client jupyter-ui-poll
    # Ensure the newest version of the helpers package is installed
    %pip uninstall -y dtc_is_notebooks
    %pip install -q --upgrade --force-reinstall --no-deps --no-cache-dir  git+https://github.com/DTC-Ice-Sheets/dtc_is_notebooks.git
    # The interactive matplotlib map (ipympl) needs Colab's custom widget manager
    from google.colab import output

    output.enable_custom_widget_manager()
    print("Dependencies installed.")

In [ ]:
%matplotlib widget
import sys
from datetime import date, datetime

import ipywidgets as widgets
import plotly.io as pio
import xarray as xr
from dtc_query_client import SuperResolutionApi, SuperResolutionRcmEmulatorLocationRequest
from dtc_query_client.helpers import wait_for_job
from IPython.display import display as ipydisplay

from dtc_is_notebook_helpers.authentication_widget import request_api_token
from dtc_is_notebook_helpers.cell_spinner import enable_cell_spinner, set_message
from dtc_is_notebook_helpers.uc4_plotting_helpers import (
    build_explorer,
    plot_single_date,
    plot_validation,
    select_location,
)

# Show an animation at the bottom of each cell while it is running
enable_cell_spinner()

if "google.colab" in sys.modules:
    pio.renderers.default = "colab"

### Enter DTC-IS API token

Please run the cell below, paste your DTC-IS API token into the box, and press 'Submit'. If you don’t have a token yet, you’ll need to generate one first using the link next to the box. A quick test query will check the token, and once it passes you can move on to the next cell! To submit a different token you will need to run the cell again.

---

In [ ]:
token_widget = request_api_token()
await token_widget.wait_until_authenticated()
client = token_widget.get_client()


## Select a location

Click on the map to choose the centre of the tile to analyse.

In [ ]:
selected = select_location()

## Select time window for the analysis

Choose the emulator model and the period to predict. ERA5 monthly means are available from 1940 to the present; the RACMO reference used for validation covers a shorter period. The values are read when you run the emulator below, so you can change them and re-run that cell without re-running this one.

In [ ]:
model_options = [
    ("Baseline UNet", "model_baseline_unet"),
    ("UNet with attention", "model_attention_unet"),
]

model_dropdown = widgets.Dropdown(
    options=model_options,
    value="model_baseline_unet",
    description="Model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="380px"),
)
start_date_picker = widgets.DatePicker(
    value=date(2021, 1, 1), description="Start date:", style={"description_width": "initial"}
)
end_date_picker = widgets.DatePicker(
    value=date(2021, 6, 15), description="End date:", style={"description_width": "initial"}
)

ipydisplay(model_dropdown, start_date_picker, end_date_picker)

## Run the Super Resolution RCM emulator entrypoint

The entrypoint will retrieve the input data through the API, and perform the following operations:
- Create a tile around the selected point -> the size is coherent with the model constraints
- Run the UNet model, taking as input 8 ERA5 variables over the selected tile
- Return the input ERA5, the model's predictions, and the target RACMO data if present

In [ ]:
start_date, end_date = start_date_picker.value, end_date_picker.value
if start_date is None or end_date is None:
    raise ValueError("Please pick both a start and an end date above.")
if start_date >= end_date:
    raise ValueError("The start date must be before the end date.")
if end_date > date.today():
    raise ValueError("The end date cannot be in the future.")
lat, lon = selected["lat"], selected["lon"]
print(f"Running {model_dropdown.label} at ({lat:.4f}, {lon:.4f}) from {start_date} to {end_date}")

set_message("Submitting the emulator job...")
res = await SuperResolutionApi(client).submit_rcm_emulator_location_job(
    SuperResolutionRcmEmulatorLocationRequest(
        start_date=start_date,
        end_date=end_date,
        latitude=lat,
        longitude=lon,
        model_name=model_dropdown.value,
    )
)
job_id = res.job_id

set_message(f"Waiting for job {job_id} to complete (this may take a few minutes)...")
result = await wait_for_job(client, job_id)

print(f"Job completed with status {result.status}")
print("Outputs:")
print(result.outputs)

In [ ]:
set_message("Opening the output datasets...")
input_era5_ds = xr.open_dataset(result.outputs["data-download"]["output_path"][0])
groud_truth_racmo_ds = xr.open_dataset(result.outputs["data-download"]["output_path"][1])
out_ds = xr.open_dataset(result.outputs["rcm-emulator"]["output_path"])

## SMB predictions Explorer

This interactive explorer helps you visualize the results.

On the first row it shows one of the 8 input variables (total precipitation), the emulator's output and the RACMO reference data if available.  
Below, the timeseries shows the average value across the tile for each timestep within the selected window.

In [ ]:
build_explorer(out_ds, stride=1).show()

## Visualize the results by date

In [ ]:
DATE = datetime(2021, 3, 21)

In [ ]:
fig = plot_single_date(out_ds, DATE)
fig

## Validation - if corresponding RACMO is available

In [ ]:
DATE = datetime(2021, 3, 21)

In [ ]:
validation_fig = plot_validation(out_ds, groud_truth_racmo_ds, DATE)
validation_fig